In [ ]:
import os
import sqlite3
import operator
from typing import Annotated

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
from typing_extensions import TypedDict

In [ ]:
load_dotenv()

GroqApiKey = os.getenv("GroqAPI")
if not GroqApiKey:
    raise ValueError("GroqAPI key not found in .env file")

LLM = ChatGroq(
    api_key = GroqApiKey,
    model = "llama-3.3-70b-versatile",
    temperature = 0.6,
)

DB_FILE = "chat_history.db"

In [ ]:
class ChatState(TypedDict):
    Messages: Annotated[list[BaseMessage], operator.add]

In [ ]:
def chatNode(State: ChatState) -> dict:
   
    Response = LLM.invoke(State["Messages"])
    return {"Messages": [AIMessage(content=Response.content)]}

In [ ]:
GraphBuilder = StateGraph(ChatState)

GraphBuilder.add_node("chat_node", chatNode)
GraphBuilder.add_edge(START, "chat_node")
GraphBuilder.add_edge("chat_node", END)

Checkpointer = SqliteSaver.from_conn_string(DB_FILE)
App = GraphBuilder.compile(checkpointer=Checkpointer)

In [ ]:
def makeConfig(ThreadId: str) -> dict:
    return {"configurable": {"thread_id": ThreadId}}

def getMessageCount(ThreadId: str) -> int:
    Config = makeConfig(ThreadId)
    CurrentState = App.get_state(Config)
    return len(CurrentState.values.get("Messages", []))

def printHistory(ThreadId: str):
  
    Config = makeConfig(ThreadId)
    CurrentState  = App.get_state(Config)
    SavedMessages = CurrentState.values.get("Messages", [])

    if not SavedMessages:
        print("No conversation history for this thread")
        return

    print(f"\n\tConversation history - thread '{ThreadId}'")
    for Msg in SavedMessages:
        if isinstance(Msg, HumanMessage):
            print(f"You: {Msg.content}")
        elif isinstance(Msg, AIMessage):
            print(f"AI: {Msg.content}")

In [ ]:
def listAllThreads():
  
    if not os.path.exists(DB_FILE):
        print("Database file does not exist yet, no sessions saved")
        return

    Connection = sqlite3.connect(DB_FILE)
    Cursor = Connection.cursor()

    try:
        Cursor.execute("SELECT DISTINCT thread_id FROM checkpoints ORDER BY thread_id")
        Rows = Cursor.fetchall()

        if not Rows:
            print("No threads stored in the database yet")
        else:
            print(f"\n\tActive threads in '{DB_FILE}'")
            for Index, Row in enumerate(Rows, start=1):
                Count = getMessageCount(Row[0])
                print(f"{Index}. {Row[0]} ({Count} messages)")

    except sqlite3.OperationalError:
        print("No threads stored in the database yet")
    finally:
        Connection.close()

In [ ]:
print("\n\tPersistent Chatbot\n")
print("Returning user: enter your existing thread ID.")
print("New user: enter any new ID")
print()

ActiveThreadId = input("Thread ID: ").strip()

if not ActiveThreadId:
    raise ValueError("Thread ID cannot be empty.")

ActiveConfig = makeConfig(ActiveThreadId)
MsgCount = getMessageCount(ActiveThreadId)

if MsgCount > 0:
    print(f"\nResuming session '{ActiveThreadId}'  ({MsgCount} messages on record).")
    printHistory(ActiveThreadId)
else:
    print(f"\nStarting new session '{ActiveThreadId}'.")

print("Commands: /history  /threads  /quit")

while True:
    UserInput = input("You: ").strip()

    if not UserInput:
        continue

    if UserInput.lower() == "/quit":
        print("Session ended. Your conversation is saved and can be resumed anytime.")
        break

    if UserInput.lower() == "/history":
        printHistory(ActiveThreadId)
        continue

    if UserInput.lower() == "/threads":
        listAllThreads()
        continue

    NewMessage = {"Messages": [HumanMessage(content=UserInput)]}
    Result = App.invoke(NewMessage, config=ActiveConfig)

    AiReply = Result["Messages"][-1].content
    print(f"AI: {AiReply}\n")

In [ ]:
InspectThreadId = input("Enter a thread ID to inspect its checkpoints: ").strip()

InspectConfig = makeConfig(InspectThreadId)
CheckpointList = list(App.get_state_history(InspectConfig))

print(f"\nTotal checkpoints for thread '{InspectThreadId}': {len(CheckpointList)}")

for Index, Snapshot in enumerate(CheckpointList):
    MsgCount = len(Snapshot.values.get("Messages", []))
    NextNode = Snapshot.next
    CheckpointId = Snapshot.config["configurable"].get("checkpoint_id", "N/A")
    print(f"Snapshot {Index + 1}: {MsgCount} message , next={NextNode} , checkpoint_id={CheckpointId}")